In [1]:
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

diem      = np.load("/content/drive/MyDrive/python-learning/data/diem.npy")
thu_nhap  = np.load("/content/drive/MyDrive/python-learning/data/thu_nhap.npy")
du_no     = np.load("/content/drive/MyDrive/python-learning/data/du_no.npy")
nhom_no   = np.load("/content/drive/MyDrive/python-learning/data/nhom_no.npy")

Mounted at /content/drive


In [22]:
import numpy as np

#(a) Tìm top 10 khách điểm cao nhất — in ra điểm VÀ chỉ số gốc của họ trong array
top10_diem_idx = np.argsort(diem)[-10:]
top10_diem = diem[top10_diem_idx]
print(f"Vị trí top 10 trong danh sách: {top10_diem_idx}")
print(f"Điểm top 10: {top10_diem}")

#(b) Ghép diem, thu_nhap, du_no thành một ma trận shape (500, 3)
data = np.column_stack((diem, thu_nhap, du_no))
print(f"Ma trận data: {data}")

#(c) Chia ngẫu nhiên seed=42: 70% train, 30% test — kiểm tra tổng số hàng train + test = 500
np.random.seed(42)
idx = np.random.permutation(len(data))

n_train = int(0.7 * len(idx))
train_idx = idx[:n_train]
test_idx  = idx[n_train:]

train_data = data[train_idx]
test_data = data[test_idx]

print(len(train_idx) + len(test_idx))


#print(train_data)
#print(test_data)

Vị trí top 10 trong danh sách: [125 220 209 179 284 234 378 374 478 113]
Điểm top 10: [850. 850. 850. 850. 850. 850. 850. 850. 850. 850.]
Ma trận data: [[7.19737132e+02 3.19154039e+07 8.37412911e+07]
 [6.68938856e+02 5.21804717e+07 5.22839202e+07]
 [7.31815083e+02 9.98132865e+06 3.18215125e+08]
 ...
 [6.64772906e+02 2.76720099e+07 3.40113890e+08]
 [6.09950540e+02 1.50957086e+07 2.08364503e+07]
 [5.69376022e+02 2.67434983e+07 4.23960080e+07]]
500


In [2]:
import numpy as np
np.random.seed(42)
n = 500

data = np.column_stack([diem, thu_nhap, du_no])

#Bước 1 — Kiểm tra NaN và điền bằng median từng cột:

check_nan = np.sum(np.isnan(data), axis = 0)
median_nan = np.nanmedian(data, axis = 0)

for i in range(data.shape[1]):
    mask = np.isnan(data[:, i])
    data[mask, i] = median_nan[i]

#print(f"Số lượng nan từng cột: {check_nan}")
#print(f"Median từng cột: {median_nan}")


#Bước 2 — Phát hiện outlier bằng IQR, đếm số lượng
q1 = np.percentile(data, 25, axis = 0)
q3 = np.percentile(data, 75, axis = 0)
iqr = q3 - q1
lower = np.maximum(0, q1 - iqr*1.5)
upper = q3 + iqr*1.5
so_outliers = (data < lower) | (data > upper)


# so_outliers shape (500, 3) — True = ô đó là outlier
# np.any(axis=1) → "hàng nào có ÍT NHẤT 1 True?"
outlier_mask = np.any(so_outliers, axis=1)   # shape (500,)

#print(f"Số hàng có outlier: {outlier_mask.sum()}")

# Lọc bỏ — dùng ~ để đảo mask
data_sach = data[~outlier_mask]

#print(f"Trước: {data.shape[0]} hàng")
#print(f"Sau:   {data_sach.shape[0]} hàng")



#Bước 3 — Chuẩn hóa Z-score:
# (x - mean) / std cho từng cột
# Dùng broadcasting — không viết 3 lần cho 3 cột
# Kiểm tra: mean của mỗi cột ≈ 0, std ≈ 1
mean_data_sach = np.mean(data_sach, axis = 0)
std_data_sach = np.std(data_sach, axis = 0)
z_score = (data_sach - mean_data_sach)/std_data_sach
print(z_score)
#print(f"Mean từng cột: {np.mean(z_score, axis = 0).round(6)}")
#print(f"Std  từng cột: {np.std(z_score, axis = 0).round(6)}")

#Bước 4 — Chia train / val / test = 70% / 15% / 15%:
# seed=42, dùng permutation
# Kiểm tra: tổng 3 tập = số hàng sau khi lọc outlier
np.random.seed(42)
idx = np.random.permutation(len(data_sach))
#print(idx)
n_train = int(0.7 * len(data_sach))
n_val = int(0.15 * len(data_sach))

train = data_sach[idx[:n_train]]
val   = data_sach[idx[n_train : n_train + n_val]]
test  = data_sach[idx[n_train + n_val:]]

mean_train = np.mean(train, axis = 0)
std_train = np.std(train, axis = 0)

z_score_train = (train - mean_train)/ std_train
z_score_val = (val - mean_train)/ std_train
z_score_test = (test - mean_train)/ std_train

#print(z_score_train)
#print(z_score_val)
#print(z_score_test)

for ten, tap in [("Train", z_score_train),
                 ("Val",   z_score_val),
                 ("Test",  z_score_test)]:
    print(f"\n{ten}:")
    print(f"  Shape : {tap.shape}")
    print(f"  Mean  : {np.mean(tap, axis=0).round(4)}")
    print(f"  Std   : {np.std(tap, axis=0).round(4)}")

[[ 0.50383764  1.20079839 -1.14948156]
 [ 0.66259631 -1.24811085  0.46420448]
 [ 1.58307047  0.60904429  0.70523652]
 ...
 [-0.21863983  0.72702931  0.61491525]
 [-0.93925261 -0.67709738 -1.58240264]
 [-1.47258451  0.62336227 -1.43402635]]

Train:
  Shape : (331, 3)
  Mean  : [ 0. -0.  0.]
  Std   : [1. 1. 1.]

Val:
  Shape : (71, 3)
  Mean  : [-0.1377  0.107   0.1876]
  Std   : [0.9689 0.9926 0.9563]

Test:
  Shape : (72, 3)
  Mean  : [ 0.1363 -0.0621 -0.0111]
  Std   : [1.0578 0.8046 0.9456]


In [42]:
import numpy as np

#(a) Top 50 điểm cao nhất → tính điểm TB và thu nhập TB của nhóm này
top_50_diem_idx = np.argsort(diem)[-50:]
top_50_diem = diem[top_50_diem_idx]
top_50_tn = thu_nhap[top_50_diem_idx]
top_50_dn = du_no[top_50_diem_idx]
mean_diem_top_50 = np.mean(top_50_diem)
mean_tn_top_50 = np.mean(top_50_tn)
mean_dn_top_50 = np.mean(top_50_dn)

#(b) Bottom 50 điểm thấp nhất → tính điểm TB và thu nhập TB
bottom_50_diem_idx = np.argsort(diem)[:50]
bottom_50_diem = diem[bottom_50_diem_idx]
bottom_50_tn = thu_nhap[bottom_50_diem_idx]
bottom_50_dn = du_no[bottom_50_diem_idx]
mean_diem_bottom_50 = np.mean(bottom_50_diem)
mean_tn_bottom_50 = np.mean(bottom_50_tn)
mean_dn_bottom_50 = np.mean(bottom_50_dn)

#(c) In bảng so sánh 2 nhóm — điểm TB, thu nhập TB, dư nợ TB
print(f"Bảng so sánh thông tin:")
print(f"Chỉ tiêu      Trung bình điểm        Trung bình thu nhập    Trung bình dư nợ")
print(f"Top 50        {mean_diem_top_50:.0f} {'':>20}{mean_tn_top_50:,.0f} {'':>11}{mean_dn_top_50:,.0f}")
print(f"Bottom 50     {mean_diem_bottom_50:.0f}{'':>21}{mean_tn_bottom_50:,.0f}{'':>12}{mean_dn_bottom_50:,.0f}")
print('-'*80)
print(f"Chênh lệch    {(mean_diem_top_50 - mean_diem_bottom_50):.0f}{'':>21}{(mean_tn_top_50 - mean_tn_bottom_50):,.0f}{'':>12}{(mean_dn_top_50 - mean_dn_bottom_50):,.0f}")


Bảng so sánh thông tin:
Chỉ tiêu      Trung bình điểm        Trung bình thu nhập    Trung bình dư nợ
Top 50        818                     22,357,509            264,003,411
Bottom 50     549                     24,910,880            255,910,135
--------------------------------------------------------------------------------
Chênh lệch    269                     -2,553,371            8,093,276


In [43]:
import numpy as np

np.random.seed(42)
n_simulations = 1000
n_customers   = 500
ty_le_vo_no   = np.zeros(n_simulations)

# Mỗi vòng lặp = 1 lần mô phỏng
# Tạo 500 điểm ngẫu nhiên, đếm tỷ lệ < 600
for i in range(n_simulations):
    diem_sim = np.random.normal(680, 80, n_customers)
    ty_le_vo_no[i] = np.mean(diem_sim < 600)

# Sau 1000 lần — phân tích phân phối tỷ lệ vỡ nợ
print(f"P5  (trường hợp tốt):  {np.percentile(ty_le_vo_no, 5):.1%}")
print(f"P50 (trung vị):        {np.percentile(ty_le_vo_no, 50):.1%}")
print(f"P95 (trường hợp xấu): {np.percentile(ty_le_vo_no, 95):.1%}")

P5  (trường hợp tốt):  13.2%
P50 (trung vị):        15.8%
P95 (trường hợp xấu): 18.6%
